# High School Stats to Recruit Matcher

Match high school players from cleaned stat files to college football recruits using 2 layers:
1. **Layer 1**: Exact name + exact state + fuzzy school (≥80%), recruit year 1-3 years ahead
2. **Layer 2**: Name variations (via nickname map) + fuzzy name (≥90%) + exact state + fuzzy school (≥80%), recruit year 1-3 years ahead

School names cleaned by removing "HS" and "High School" indicators before matching.

## Setup: Import Libraries

In [48]:
import pandas as pd
import os
import re
import time
import difflib
from pathlib import Path

# Install fuzzywuzzy if not already installed
try:
    from fuzzywuzzy import fuzz, process
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'fuzzywuzzy', 'python-Levenshtein'])
    from fuzzywuzzy import fuzz, process

print("Libraries loaded successfully")

# Set base directories for relative path resolution
BASE_PROJECT_DIR = r"X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence"
BASE_DATA_DIR = os.path.join(BASE_PROJECT_DIR, 'data')
HS_STATS_DIR = os.path.join(BASE_DATA_DIR, 'high_school_stats','CSV')
RECRUITS_DIR = os.path.join(BASE_DATA_DIR, 'Football Recruitment Tables','Recruitment Classes')
NAME_MATCH_DIR = os.path.join(BASE_DATA_DIR, 'name_matching','hs_recruit')

print(f"Base data directory: {os.path.abspath(BASE_DATA_DIR)}")
print(f"Base project directory: {os.path.abspath(BASE_PROJECT_DIR)}")
print(f"High school stats directory: {os.path.abspath(HS_STATS_DIR)}")
print(f"Recruits directory: {os.path.abspath(RECRUITS_DIR)}")

Libraries loaded successfully
Base data directory: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data
Base project directory: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence
High school stats directory: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\high_school_stats\CSV
Recruits directory: X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\Recruitment Classes


In [64]:
import json

def clean_name(name):
    """
    Cleans player/team names for matching:
    - Removes special characters and apostrophes
    - Normalizes spacing
    - Converts to lowercase for comparison
    """
    if not name or pd.isna(name):
        return ""
    name = str(name).lower().strip()
    # Remove special characters except spaces and hyphens
    name = re.sub(r"[^a-z0-9\s\-]", "", name)
    # Normalize multiple spaces
    name = re.sub(r"\s+", " ", name).strip()
    return name

def split_name(name_clean):
    """
    Splits a cleaned name into first and last components.
    Returns tuple (first_name, last_name).
    """
    if not name_clean or len(name_clean.split()) == 0:
        return name_clean, ""
    
    parts = name_clean.split()
    if len(parts) == 1:
        return parts[0], ""
    else:
        return parts[0], ' '.join(parts[1:])

def extract_quoted_nickname(raw_name):
    """
    Extracts quoted nicknames from a raw name.
    Example: "Christopher 'Chris' Jones" → ('Christopher', 'Chris', 'Jones')
    Returns: (formal_first, quoted_nickname, last) where quoted_nickname is None if not found
    """
    if not raw_name or pd.isna(raw_name):
        return None, None
    
    # Look for pattern: word 'word' word
    match = re.search(r"(\w+)\s+['\"](\w+)['\"]\s+", str(raw_name))
    if match:
        formal_first = match.group(1).lower()
        quoted_nick = match.group(2).lower()
        return formal_first, quoted_nick
    
    return None, None

def expand_name_variations(name_clean, nickname_map, quoted_nickname=None):
    """
    Expands a cleaned name with nickname variations from the nickname map + quoted nicknames.
    Returns a list of name variations (including original).
    Example: 'william' → ['william', 'bill', 'billy', 'will', 'liam']
    If quoted_nickname provided: also includes quoted variation
    """
    variations = [name_clean]  # Always include original
    
    # Add quoted nickname if provided
    if quoted_nickname:
        variations.append(quoted_nickname)
    
    # Check if this name has known nicknames
    if name_clean in nickname_map:
        variations.extend(nickname_map[name_clean])
    
    # Check if quoted nickname has known nicknames
    if quoted_nickname and quoted_nickname in nickname_map:
        variations.extend(nickname_map[quoted_nickname])
    
    # Check if this name is a nickname for something else (reverse lookup)
    for full_name, nicknames in nickname_map.items():
        if name_clean in nicknames:
            variations.append(full_name)
        if quoted_nickname and quoted_nickname in nicknames:
            variations.append(full_name)
    
    return list(set(variations))  # Remove duplicates

# Load nickname mapping from JSON
nickname_map_path = os.path.join(BASE_DATA_DIR, "name_matching", "nickname_map.json")
try:
    with open(nickname_map_path, 'r') as f:
        nickname_data = json.load(f)
        nickname_map = nickname_data.get('nicknames', {})
    print(f"✓ Loaded nickname map with {len(nickname_map)} entries")
except Exception as e:
    print(f"WARNING: Could not load nickname map: {e}")
    nickname_map = {}

print("Name cleaning and nickname expansion functions defined")

✓ Loaded nickname map with 359 entries
Name cleaning and nickname expansion functions defined


## Load High School Stats CSV Files and Create Unique Player List


In [50]:
import pandas as pd
import os
import glob
from pathlib import Path

# Use the directory variables from the setup cell above
print("=" * 80)
print("LOADING HIGH SCHOOL STATS CSVs")
print("=" * 80)

# Find all CSV files recursively in the CSV directory
csv_files = glob.glob(os.path.join(HS_STATS_DIR, "**", "*.csv"), recursive=True)
print(f"Found {len(csv_files)} CSV files\n")

if not csv_files:
    print(f"ERROR: No CSV files found in {HS_STATS_DIR}")
else:
    # Load and combine all CSV files
    all_data = []
    
    for csv_file in sorted(csv_files):
        try:
            df = pd.read_csv(csv_file, encoding='utf-8')
            filename = os.path.basename(csv_file)
            print(f"  ✓ Loaded {filename}: {len(df)} records")
            all_data.append(df)
        except Exception as e:
            print(f"  ✗ Error loading {csv_file}: {e}")
    
    # Combine all dataframes
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n{'='*80}")
        print(f"Total records loaded: {len(combined_df)}")
        print(f"Columns available: {list(combined_df.columns)}")
        print(f"{'='*80}\n")
        
        # Check for career_id column (renamed from careerId in column_map)
        if 'career_id' not in combined_df.columns:
            print("WARNING: 'career_id' column not found in data")
            print(f"Available columns: {combined_df.columns.tolist()}")
        else:
            # Filter out rows with empty career_id
            valid_df = combined_df[combined_df['career_id'].notna() & (combined_df['career_id'] != '')]
            print(f"Records with valid career_id: {len(valid_df)} / {len(combined_df)}")
            
            # Get required columns (using renamed column names from column_map)
            required_cols = ['full_name', 'position_category', 'career_id', 'year', 'school_name', 'school_state']
            missing_cols = [col for col in required_cols if col not in valid_df.columns]
            
            if missing_cols:
                print(f"WARNING: Missing columns: {missing_cols}")
                print(f"Available columns: {valid_df.columns.tolist()}")
            else:
                # Create a dataframe with athlete name, position, career_id, school info from final year
                # Sort by year and get the last row per career_id to get final year school info
                final_year_info = valid_df.sort_values('year').groupby('career_id').tail(1)[['career_id', 'school_name', 'school_state']]
                
                # Get name, position, and final year
                unique_players = valid_df.groupby('career_id').agg({
                    'full_name': 'first',  # Get name once per career_id
                    'position_category': 'first',  # Get position once per career_id
                    'year': 'max',  # Get maximum (last) year in data
                }).reset_index()
                
                # Merge in school info from the final year
                unique_players = unique_players.merge(final_year_info, on='career_id', how='left')
                
                # Add cleaned name
                unique_players['name_cleaned'] = unique_players['full_name'].apply(clean_name)
                
                # Rename columns for clarity
                unique_players = unique_players.rename(columns={
                    'full_name': 'name',
                    'position_category': 'position',
                    'year': 'final_year'
                })
                
                # Clean school names: remove "HS" and "High School" variants
                def clean_school_name(school):
                    if not school or pd.isna(school):
                        return ""
                    s = str(school).strip()
                    # Remove common HS/High School indicators
                    s = re.sub(r'\bHS\b', '', s, flags=re.IGNORECASE)
                    s = re.sub(r'\bHigh School\b', '', s, flags=re.IGNORECASE)
                    s = re.sub(r'\bH\.S\.\b', '', s, flags=re.IGNORECASE)
                    # Normalize whitespace
                    s = re.sub(r'\s+', ' ', s).strip()
                    return s
                
                unique_players['school_name'] = unique_players['school_name'].apply(clean_school_name)
                
                # Reorder columns
                unique_players = unique_players[['name', 'name_cleaned', 'career_id', 'position', 'final_year', 'school_name', 'school_state']]
                
                print(f"\n{'='*80}")
                print(f"UNIQUE PLAYERS BY CAREER ID (Schools cleaned)")
                print(f"{'='*80}")
                print(f"Total unique players: {len(unique_players)}")
                print(f"\nSample data:")
                print(unique_players.head(10).to_string(index=False))
                print(f"\nPosition breakdown:")
                print(unique_players['position'].value_counts())
                print(f"\nFinal year in data:")
                print(unique_players['final_year'].value_counts().sort_index(ascending=False))
    else:
        print("ERROR: Could not load any CSV files")
        
# Save unique players to CSV
OUTPUT_DIR = NAME_MATCH_DIR
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "unique_hs_players.csv")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

unique_players.to_csv(OUTPUT_FILE, index=False)
print(f"Unique players saved to {OUTPUT_FILE}")

LOADING HIGH SCHOOL STATS CSVs
Found 105 CSV files

  ✓ Loaded hs_interceptions_2011.csv: 500 records
  ✓ Loaded hs_interceptions_2012.csv: 500 records
  ✓ Loaded hs_interceptions_2013.csv: 500 records
  ✓ Loaded hs_interceptions_2014.csv: 500 records
  ✓ Loaded hs_interceptions_2015.csv: 500 records
  ✓ Loaded hs_interceptions_2016.csv: 500 records
  ✓ Loaded hs_interceptions_2017.csv: 500 records
  ✓ Loaded hs_interceptions_2018.csv: 500 records
  ✓ Loaded hs_interceptions_2019.csv: 500 records
  ✓ Loaded hs_interceptions_2020.csv: 500 records
  ✓ Loaded hs_interceptions_2021.csv: 500 records
  ✓ Loaded hs_interceptions_2022.csv: 500 records
  ✓ Loaded hs_interceptions_2023.csv: 500 records
  ✓ Loaded hs_interceptions_2024.csv: 500 records
  ✓ Loaded hs_interceptions_2025.csv: 500 records
  ✓ Loaded hs_passing_2011.csv: 500 records
  ✓ Loaded hs_passing_2012.csv: 500 records
  ✓ Loaded hs_passing_2013.csv: 500 records
  ✓ Loaded hs_passing_2014.csv: 500 records
  ✓ Loaded hs_passing_

## Create Recruit List

In [51]:
import glob
from pathlib import Path

print("=" * 80)
print("LOADING ALL RECRUIT CLASSES")
print("=" * 80)

# Find all recruit CSVs
recruit_files = sorted(glob.glob(os.path.join(RECRUITS_DIR, "recruits_*.csv")))
print(f"Found {len(recruit_files)} recruit files\n")

all_recruits = []

# Function to clean school names
def clean_school_name(school):
    if not school or pd.isna(school):
        return ""
    s = str(school).strip()
    # Remove common HS/High School indicators
    s = re.sub(r'\bHS\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bHigh School\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\bH\.S\.\b', '', s, flags=re.IGNORECASE)
    # Normalize whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

for recruit_file in recruit_files:
    try:
        df = pd.read_csv(recruit_file)
        filename = os.path.basename(recruit_file)
        
        # Extract only needed columns
        if all(col in df.columns for col in ['year', 'name', 'high_school', 'state', 'player_id', 'url']):
            subset = df[['year', 'name', 'high_school', 'state', 'player_id', 'url']].copy()
            
            # Add cleaned name
            subset['name_cleaned'] = subset['name'].apply(clean_name)
            
            # Clean school names: remove "HS" and "High School" variants
            subset['high_school'] = subset['high_school'].apply(clean_school_name)
            
            # Extract player_id from URL (last part before final /)
            # URL format: https://247sports.com/player/name-slug-12345/
            subset['player_id_from_url'] = subset['url'].str.extract(r'(\d+)/?$')[0]
            
            all_recruits.append(subset)
            print(f"  ✓ Loaded {len(subset)} recruits from {filename}")
    except Exception as e:
        print(f"  ✗ Error loading {filename}: {e}")

# Combine all recruit dataframes
if all_recruits:
    recruits_compiled = pd.concat(all_recruits, ignore_index=True)
    
    # Reorder columns for clarity
    recruits_compiled = recruits_compiled[['year', 'name', 'name_cleaned', 'high_school', 'state', 'player_id', 'player_id_from_url', 'url']]
    
    print(f"\n{'='*80}")
    print(f"COMPILED RECRUITS SUMMARY (Schools cleaned)")
    print(f"{'='*80}")
    print(f"Total recruits: {len(recruits_compiled)}")
    print(f"Years covered: {sorted(recruits_compiled['year'].unique())}")
    print(f"States represented: {recruits_compiled['state'].nunique()}")
    print(f"\nSample data:")
    print(recruits_compiled.head(10))
    
    # Save compiled recruits to CSV
    OUTPUT_DIR = NAME_MATCH_DIR
    OUTPUT_FILE = os.path.join(OUTPUT_DIR, "compiled_recruit_players.csv")
    
    # Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    recruits_compiled.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✓ Compiled recruits saved to {OUTPUT_FILE}")
else:
    print("ERROR: Could not load any recruit files")
    recruits_compiled = None


LOADING ALL RECRUIT CLASSES
Found 14 recruit files

  ✓ Loaded 3518 recruits from recruits_2015.csv
  ✓ Loaded 3942 recruits from recruits_2016.csv


  ✓ Loaded 4212 recruits from recruits_2017.csv
  ✓ Loaded 3886 recruits from recruits_2018.csv
  ✓ Loaded 3986 recruits from recruits_2019.csv
  ✓ Loaded 3862 recruits from recruits_2020.csv
  ✓ Loaded 2665 recruits from recruits_2021.csv
  ✓ Loaded 2194 recruits from recruits_2022.csv
  ✓ Loaded 2294 recruits from recruits_2023.csv
  ✓ Loaded 3652 recruits from recruits_2024.csv
  ✓ Loaded 3566 recruits from recruits_2025.csv
  ✓ Loaded 2933 recruits from recruits_2026.csv
  ✓ Loaded 989 recruits from recruits_2027.csv
  ✓ Loaded 227 recruits from recruits_2028.csv

COMPILED RECRUITS SUMMARY (Schools cleaned)
Total recruits: 41926
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028)]
States represented: 81

Sample data:
   year              name      name_cleaned                high_school state  \
0 

In [52]:
"""
OPTIMIZATION: Build two bucketing indices - strict for Layer 1, relaxed for Layers 2-3
Layer 1: (state, first_letter) - checks all recruit years in the 1-3 year lead window
Layers 2-3: (year, state) - for fuzzy matching with flexible tolerances
"""

print("=" * 80)
print("PRE-PROCESSING: BUILD STRICT AND RELAXED BUCKETING INDICES")
print("=" * 80)

# Pre-compute cleaned states for recruits (avoid cleaning repeatedly)
recruits_compiled['state_cleaned'] = recruits_compiled['state'].apply(
    lambda x: clean_name(str(x)) if pd.notna(x) else ''
)

# Pre-compute cleaned schools for recruits
recruits_compiled['high_school_cleaned'] = recruits_compiled['high_school'].apply(
    lambda x: clean_name(str(x)) if pd.notna(x) else ''
)

# Pre-compute cleaned states for HS players
unique_players['school_state_cleaned'] = unique_players['school_state'].apply(
    lambda x: clean_name(str(x)) if pd.notna(x) else ''
)

# Pre-compute cleaned schools for HS players
unique_players['school_name_cleaned'] = unique_players['school_name'].apply(
    lambda x: clean_name(str(x)) if pd.notna(x) else ''
)

print("✓ Pre-computed cleaned school/state names for all records")

# =====================================================================================
# INDEX 1: STRICT bucketing for Layer 1 (Generous year window via matching logic)
# Key: (state, first_letter) - includes all years, will filter by 1-3 year lead in matching
# =====================================================================================
recruit_index_strict = {}

for idx, recruit_row in recruits_compiled.iterrows():
    try:
        year = int(recruit_row['year'])
        name_clean = recruit_row['name_cleaned']
        state_clean = recruit_row['state_cleaned']
        
        # Skip records with missing critical fields
        if not name_clean or not state_clean:
            continue
        
        first_letter = name_clean[0]
        # Key: (state, first_letter) - year will be checked during matching
        key = (state_clean, first_letter)
        
        if key not in recruit_index_strict:
            recruit_index_strict[key] = []
        
        recruit_index_strict[key].append(recruit_row)
    except (ValueError, TypeError, KeyError):
        continue

print(f"✓ Built STRICT recruit index (Layer 1): {len(recruit_index_strict)} buckets")
print(f"  Key: (state, first_letter) - will check recruit year 1-3 years ahead of HS year")

# =====================================================================================
# INDEX 2: RELAXED bucketing for Layers 2-3 (Fuzzy matching)
# Key: (year, state) - includes all first_letters, much more permissive
# =====================================================================================
recruit_index_relaxed = {}

for idx, recruit_row in recruits_compiled.iterrows():
    try:
        year = int(recruit_row['year'])
        state_clean = recruit_row['state_cleaned']
        
        if not state_clean:
            continue
        
        # Relaxed key: just year and state (no first_letter constraint)
        key = (year, state_clean)
        
        if key not in recruit_index_relaxed:
            recruit_index_relaxed[key] = []
        
        recruit_index_relaxed[key].append(recruit_row)
    except (ValueError, TypeError, KeyError):
        continue

print(f"✓ Built RELAXED recruit index (Layers 2-3): {len(recruit_index_relaxed)} buckets")

# Show index statistics
bucket_sizes_strict = [len(candidates) for candidates in recruit_index_strict.values()]
bucket_sizes_relaxed = [len(candidates) for candidates in recruit_index_relaxed.values()]

print(f"\n  STRICT Index (Layer 1) - avg candidates per bucket: {sum(bucket_sizes_strict)/len(bucket_sizes_strict):.1f}")
print(f"  RELAXED Index (Layers 2-3) - avg candidates per bucket: {sum(bucket_sizes_relaxed)/len(bucket_sizes_relaxed):.1f}")
print(f"  Note: STRICT candidates checked against 1-3 year lead window + exact name/state + fuzzy school >80%")

PRE-PROCESSING: BUILD STRICT AND RELAXED BUCKETING INDICES
✓ Pre-computed cleaned school/state names for all records
✓ Built STRICT recruit index (Layer 1): 1240 buckets
  Key: (state, first_letter) - will check recruit year 1-3 years ahead of HS year
✓ Built RELAXED recruit index (Layers 2-3): 764 buckets

  STRICT Index (Layer 1) - avg candidates per bucket: 33.8
  RELAXED Index (Layers 2-3) - avg candidates per bucket: 54.8
  Note: STRICT candidates checked against 1-3 year lead window + exact name/state + fuzzy school >80%


## Section 3: Pre-Processing & Build Efficiency Index

## Section 4: Layer 1 - Exact Matching

In [66]:
# Layer 1: Exact Name + State + Fuzzy School Match with 1-3 Year Lead
# OPTIMIZED: Uses STRICT bucketing index (state, first_letter)

def is_year_valid(final_year, recruit_year, layer=1):
    """Check if year combination is valid based on tolerance rules"""
    recruit_year = int(recruit_year)
    final_year = int(final_year)
    
    # Special case: recruits from 2026+ should match HS players from 2025 (data cutoff)
    if recruit_year >= 2026 and final_year == 2025:
        return True
    
    # Layer 1: recruit year must be 1-3 years AHEAD of final HS year (generous window)
    if layer == 1:
        return final_year < recruit_year <= final_year + 3
    # Layer 2: ±1 year for nickname/fuzzy matching 
    elif layer == 2:
        tolerance = 1
        return final_year <= recruit_year <= final_year + 1 + tolerance
    # Layer 3: ±2 years for school variant matching
    elif layer == 3:
        tolerance = 2
        return final_year <= recruit_year <= final_year + 1 + tolerance
    else:  # Layer 4+
        tolerance = 3
        return final_year <= recruit_year <= final_year + 1 + tolerance


print("=" * 80)
print("LAYER 1: EXACT NAME + STATE + FUZZY SCHOOL (>80%) + YEAR 1-3 AHEAD")
print("=" * 80)

exact_matches = []
matched_hs_ids = set()  # Track matched players for Layers 2-4

start_time = time.time()
total_hs_players = len(unique_players)

for idx, hs_row in unique_players.iterrows():
    if (idx + 1) % 5000 == 0:
        print(f"  Progress: {idx + 1}/{total_hs_players} ({100*(idx+1)/total_hs_players:.1f}%)")
    
    hs_name = hs_row['name_cleaned']
    hs_school = hs_row['school_name_cleaned']
    hs_state = hs_row['school_state_cleaned']
    hs_final_year = int(hs_row['final_year'])
    hs_career_id = hs_row['career_id']
    
    # Skip invalid records
    if not hs_name or not hs_school or not hs_state:
        continue
    
    first_letter = hs_name[0]
    
    # Use STRICT index with (state, first_letter) key
    key = (hs_state, first_letter)
    
    if key not in recruit_index_strict:
        continue
    
    candidates = recruit_index_strict[key]
    
    for recruit_row in candidates:
        recruit_name = recruit_row['name_cleaned']
        recruit_school = recruit_row['high_school_cleaned']
        recruit_state = recruit_row['state_cleaned']
        recruit_year = int(recruit_row['year'])
        
        # Criteria: exact name match, exact state match, year 1-3 ahead, school fuzzy >80%
        if (hs_name == recruit_name and 
            hs_state == recruit_state and
            is_year_valid(hs_final_year, recruit_year, layer=1)):
            
            # Fuzzy match school names with 80% threshold
            school_score = fuzz.token_set_ratio(hs_school, recruit_school)
            
            if school_score >= 80:
                exact_matches.append({
                    'hs_player': hs_row['name'],
                    'hs_school': hs_row['school_name'],
                    'hs_state': hs_row['school_state'],
                    'hs_year': hs_row['final_year'],
                    'recruit_name': recruit_row['name'],
                    'recruit_school': recruit_row['high_school'],
                    'recruit_state': recruit_row['state'],
                    'recruit_year': recruit_row['year'],
                    'recruit_id': recruit_row['player_id'],
                    'match_confidence': (100.0 + school_score) / 2,
                    'matching_method': f'Exact Name + State + Fuzzy School ({school_score:.0f}%)'
                })
                matched_hs_ids.add(hs_career_id)
                break

exact_matches_df = pd.DataFrame(exact_matches) if exact_matches else pd.DataFrame()
elapsed = time.time() - start_time

print(f"\n✓ Layer 1 Complete: {len(exact_matches_df)} matches in {elapsed:.1f}s")
if len(exact_matches_df) > 0:
    print(exact_matches_df[['hs_player', 'hs_school', 'hs_state', 'recruit_name']].head(5).to_string(index=False))

LAYER 1: EXACT NAME + STATE + FUZZY SCHOOL (>80%) + YEAR 1-3 AHEAD
  Progress: 5000/38006 (13.2%)
  Progress: 10000/38006 (26.3%)
  Progress: 15000/38006 (39.5%)
  Progress: 20000/38006 (52.6%)
  Progress: 25000/38006 (65.8%)
  Progress: 30000/38006 (78.9%)
  Progress: 35000/38006 (92.1%)

✓ Layer 1 Complete: 4155 matches in 41.3s
         hs_player       hs_school hs_state       recruit_name
     Grayson Hardy Sulphur Springs       TX      Grayson Hardy
 Dominic Ingrassia       San Marin       CA  Dominic Ingrassia
     Isaiah Phelps        Pacifica       CA      Isaiah Phelps
       Brady Allen Gibson Southern       IN        Brady Allen
Jatarvious Whitlow       LaFayette       AL JaTarvious Whitlow


## Section 5: Layer 2 - Name Variations & Fuzzy Matching

In [67]:
# Layer 2: FIRST NAME (nickname variations + fuzzy 90%) + LAST NAME (exact, 95%+) + EXACT STATE + FUZZY SCHOOL (80%+)
# STRICTER: Requires last name to match almost exactly, first name can be nickname/fuzzy
# Uses STRICT bucketing with 1-3 year lead window
# Goal: Precise matches with name variations permitted only for first names

print("=" * 80)
print("LAYER 2: FIRST NAME FUZZY (90%) + LAST NAME EXACT (95%) + STATE + SCHOOL (80%)")
print("=" * 80)

fuzzy_matches = []
start_time = time.time()
total_hs_players = len(unique_players)

for idx, hs_row in unique_players.iterrows():
    if (idx + 1) % 5000 == 0:
        print(f"  Progress: {idx + 1}/{total_hs_players} ({100*(idx+1)/total_hs_players:.1f}%)")
    
    hs_career_id = hs_row['career_id']
    
    # Skip if already matched in Layer 1
    if hs_career_id in matched_hs_ids:
        continue
    
    hs_name = hs_row['name_cleaned']
    hs_school = hs_row['school_name_cleaned']
    hs_state = hs_row['school_state_cleaned']
    hs_final_year = int(hs_row['final_year'])
    
    # Skip invalid records
    if not hs_name or not hs_school or not hs_state:
        continue
    
    # Split HS name into first and last
    hs_first, hs_last = split_name(hs_name)
    if not hs_first or not hs_last:
        continue
    
    # Extract quoted nickname if present (e.g., "Christopher 'Chris' Jones")
    formal_first_raw, quoted_nick = extract_quoted_nickname(hs_row['name'])
    
    # Get first name variations (for nickname matching + quoted nicknames)
    first_name_variations = expand_name_variations(hs_first, nickname_map, quoted_nick)
    
    # Use STRICT index (state, first_letter) with year window checking
    first_letter = hs_name[0]
    key = (hs_state, first_letter)
    
    if key not in recruit_index_strict:
        continue
    
    candidates = recruit_index_strict[key]
    
    for recruit_row in candidates:
        recruit_name = recruit_row['name_cleaned']
        recruit_school = recruit_row['high_school_cleaned']
        recruit_state = recruit_row['state_cleaned']
        recruit_year = int(recruit_row['year'])
        
        # MUST: Exact state match
        if hs_state != recruit_state:
            continue
        
        # MUST: Valid year (1-3 years ahead)
        if not is_year_valid(hs_final_year, recruit_year, layer=1):
            continue
        
        # Split recruit name into first and last
        rec_first, rec_last = split_name(recruit_name)
        if not rec_first or not rec_last:
            continue
        
        # ===== LAST NAME: Require ~90% match (highly similar, not exact) =====
        last_name_score = fuzz.ratio(hs_last, rec_last)
        if last_name_score < 90:
            continue  # Last name must match with high similarity
        
        # ===== FIRST NAME: Allow fuzzy + nicknames, require 90%+ =====
        best_first_score = 0
        
        for first_var in first_name_variations:
            first_score = fuzz.ratio(first_var, rec_first)
            best_first_score = max(best_first_score, first_score)
        
        if best_first_score < 90:
            continue  # First name must be 90%+ fuzzy match
        
        # ===== SCHOOL: Fuzzy match 80%+ =====
        school_score = fuzz.token_set_ratio(hs_school, recruit_school)
        if school_score < 80:
            continue
        
        # All criteria met - add match
        # Confidence is average of first name, last name, and school scores
        avg_name_score = (best_first_score + last_name_score) / 2
        match_confidence = (avg_name_score + school_score) / 2
        
        fuzzy_matches.append({
            'hs_player': hs_row['name'],
            'hs_school': hs_row['school_name'],
            'hs_state': hs_row['school_state'],
            'hs_year': hs_row['final_year'],
            'recruit_name': recruit_row['name'],
            'recruit_school': recruit_row['high_school'],
            'recruit_state': recruit_row['state'],
            'recruit_year': recruit_row['year'],
            'recruit_id': recruit_row['player_id'],
            'match_confidence': match_confidence,
            'matching_method': f'First Name Fuzzy ({best_first_score:.0f}%) + Last Name Exact ({last_name_score:.0f}%) + School ({school_score:.0f}%)'
        })
        matched_hs_ids.add(hs_career_id)
        break  # Move to next HS player once found

fuzzy_matches_df = pd.DataFrame(fuzzy_matches) if fuzzy_matches else pd.DataFrame()
elapsed = time.time() - start_time

print(f"\n✓ Layer 2 Complete: {len(fuzzy_matches_df)} matches in {elapsed:.1f}s")
if len(fuzzy_matches_df) > 0:
    print(fuzzy_matches_df[['hs_player', 'hs_school', 'recruit_name', 'match_confidence']].head(5).to_string(index=False))

LAYER 2: FIRST NAME FUZZY (90%) + LAST NAME EXACT (95%) + STATE + SCHOOL (80%)
  Progress: 5000/38006 (13.2%)
  Progress: 10000/38006 (26.3%)
  Progress: 15000/38006 (39.5%)
  Progress: 20000/38006 (52.6%)
  Progress: 25000/38006 (65.8%)
  Progress: 30000/38006 (78.9%)
  Progress: 35000/38006 (92.1%)

✓ Layer 2 Complete: 86 matches in 41.9s
         hs_player         hs_school       recruit_name  match_confidence
Quendarion Barnett    Noxubee County Qendarrion Barnett              97.5
   Gregory Spiller       John Champe       Greg Spiller             100.0
  Joshua Bettistea         Allatoona     Josh Bettistea             100.0
    Mike McClenton         Edgewater  Michael McClenton             100.0
     Joshua Dallas Trinity Christian        Josh Dallas             100.0


## Section 6: Combine All Matches & Generate Outputs

In [68]:
# Save all matches by layer and combine results
print("\n" + "="*80)
print("SAVING RESULTS TO CSV")
print("="*80)

output_dir = NAME_MATCH_DIR

# Combine all matches (Layer 2 only, Layer 3 skipped)
dfs_to_combine = []
if len(exact_matches_df) > 0:
    dfs_to_combine.append(exact_matches_df)
if len(fuzzy_matches_df) > 0:
    dfs_to_combine.append(fuzzy_matches_df)

all_matches = pd.concat(dfs_to_combine, ignore_index=True) if dfs_to_combine else pd.DataFrame()

# Save exact matches
if len(exact_matches_df) > 0:
    exact_output = os.path.join(output_dir, "hs_to_recruit_exact_matches.csv")
    exact_matches_df.to_csv(exact_output, index=False)
    print(f"✓ Saved exact matches ({len(exact_matches_df)}) to hs_to_recruit_exact_matches.csv")
else:
    print(f"  No exact matches to save")

# Save fuzzy matches
if len(fuzzy_matches_df) > 0:
    fuzzy_output = os.path.join(output_dir, "hs_to_recruit_fuzzy_matches.csv")
    fuzzy_matches_df.to_csv(fuzzy_output, index=False)
    print(f"✓ Saved fuzzy matches ({len(fuzzy_matches_df)}) to hs_to_recruit_fuzzy_matches.csv")
else:
    print(f"  No fuzzy matches to save")

# Save combined matches
if len(all_matches) > 0:
    combined_output = os.path.join(output_dir, "hs_to_recruit_all_matches.csv")
    all_matches.to_csv(combined_output, index=False)
    print(f"✓ Saved combined matches ({len(all_matches)}) to hs_to_recruit_all_matches.csv")

print(f"\n{'='*80}")
print(f"Final Summary:")
print(f"  Total HS Players: {len(unique_players)}")
print(f"  Total Recruits: {len(recruits_compiled)}")
print(f"  Matched: {len(all_matches)} ({100*len(all_matches)/len(unique_players):.1f}%)")
print(f"{'='*80}")


SAVING RESULTS TO CSV
✓ Saved exact matches (4155) to hs_to_recruit_exact_matches.csv
✓ Saved fuzzy matches (86) to hs_to_recruit_fuzzy_matches.csv
✓ Saved combined matches (4241) to hs_to_recruit_all_matches.csv

Final Summary:
  Total HS Players: 38006
  Total Recruits: 41926
  Matched: 4241 (11.2%)


## Section 7: Find Unmatched Players

In [72]:
# Find unmatched HS players
matched_career_ids = set()
if len(exact_matches_df) > 0:
    matched_career_ids.update(exact_matches_df['hs_player'].apply(
        lambda x: unique_players[unique_players['name'] == x]['career_id'].values[0] 
        if len(unique_players[unique_players['name'] == x]) > 0 else None
    ).dropna().tolist())

unmatched_hs = []
for idx, hs_row in unique_players.iterrows():
    if hs_row['career_id'] not in matched_hs_ids:
        unmatched_hs.append({
            'hs_player': hs_row['name'],
            'hs_school': hs_row['school_name'],
            'hs_state': hs_row['school_state'],
            'hs_year': hs_row['final_year']
        })

unmatched_hs_df = pd.DataFrame(unmatched_hs) if unmatched_hs else pd.DataFrame()
print(f"\n{'='*70}")
print(f"UNMATCHED HS PLAYERS: {len(unmatched_hs_df)} / {len(unique_players)}")
print(f"{'='*70}")

# Save unmatched
output_dir = NAME_MATCH_DIR

unmatched_output = os.path.join(output_dir, "hs_to_recruit_unmatched.csv")
unmatched_hs_df.to_csv(unmatched_output, index=False)
print(f"✓ Saved unmatched players to {unmatched_output}")


UNMATCHED HS PLAYERS: 33765 / 38006
✓ Saved unmatched players to X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\name_matching\hs_recruit\hs_to_recruit_unmatched.csv


## Section 8: Quality Checks & Exploration

In [79]:
# Quality Checks & Analysis
print("\n" + "="*70)
print("MATCHING QUALITY ANALYSIS")
print("="*70)

# View match distribution by layer
print("\nMatches by Layer:")
print(f"  Layer 1 (Exact name + state + fuzzy school 80%): {len(exact_matches_df)}")
print(f"  Layer 2 (Name variations + fuzzy 90% + exact state + fuzzy 80%): {len(fuzzy_matches_df)}")

# Combine all matches
all_matches = pd.concat([exact_matches_df, fuzzy_matches_df], 
                        ignore_index=True) if len(exact_matches_df) > 0 or len(fuzzy_matches_df) > 0 else pd.DataFrame()

print(f"  Total Matches: {len(all_matches)}")
print(f"  Unique HS Players Matched: {all_matches['hs_player'].nunique() if len(all_matches) > 0 else 0}")
print(f"  Match Rate: {100*len(matched_hs_ids)/len(unique_players):.1f}%")

# View matches by year
if len(all_matches) > 0:
    print("\nMatches by HS Year:")
    print(all_matches['hs_year'].value_counts().sort_index().to_string())
    
    print("\nMatches by Method:")
    print(all_matches['matching_method'].value_counts().to_string())

# Sample exact matches
if len(exact_matches_df) > 0:
    print("\n" + "="*70)
    print("SAMPLE LAYER 1 MATCHES (Exact name + state)")
    print("="*70)
    print(exact_matches_df[['hs_player', 'hs_school', 'hs_state', 'recruit_name', 'recruit_school']].head(10).to_string(index=False))

# Sample fuzzy matches
if len(fuzzy_matches_df) > 0:
    print("\n" + "="*70)
    print("SAMPLE LAYER 2 MATCHES (Name variations + fuzzy)")
    print("="*70)
    cols = ['hs_player', 'hs_school', 'recruit_name', 'match_confidence']
    print(fuzzy_matches_df[cols].head(10).to_string(index=False))

# Sample unmatched
if len(unmatched_hs_df) > 0:
    print("\n" + "="*70)
    print("SAMPLE UNMATCHED HS PLAYERS")
    print("="*70)
    print(unmatched_hs_df.head(10).to_string(index=False))


MATCHING QUALITY ANALYSIS

Matches by Layer:
  Layer 1 (Exact name + state + fuzzy school 80%): 4155
  Layer 2 (Name variations + fuzzy 90% + exact state + fuzzy 80%): 86
  Total Matches: 4241
  Unique HS Players Matched: 4168
  Match Rate: 11.2%

Matches by HS Year:
hs_year
2012     21
2013    114
2014    254
2015    433
2016    321
2017    376
2018    432
2019    371
2020    261
2021    216
2022    230
2023    438
2024    423
2025    351

Matches by Method:
matching_method
Exact Name + State + Fuzzy School (100%)                            4112
First Name Fuzzy (100%) + Last Name Exact (100%) + School (100%)      67
Exact Name + State + Fuzzy School (87%)                                6
Exact Name + State + Fuzzy School (81%)                                5
Exact Name + State + Fuzzy School (86%)                                5
Exact Name + State + Fuzzy School (95%)                                4
Exact Name + State + Fuzzy School (80%)                                4
Exact Na

In [80]:
# Find Highest Ranked Unmatched Recruits
print("\n" + "="*80)
print("HIGHEST RANKED UNMATCHED RECRUITS")
print("="*80)

# Get all matched recruit names for comparison
matched_recruits = set(all_matches['recruit_name'].unique()) if len(all_matches) > 0 else set()

# Find unmatched recruits
unmatched_recruits = recruits_compiled[~recruits_compiled['name'].isin(matched_recruits)].copy()

if len(unmatched_recruits) > 0:
    # Extract rank from player_id (last 5 digits)
    # Format: YYYYXXXXX where YYYY = year, XXXXX = rank
    unmatched_recruits['rank'] = unmatched_recruits['player_id'].astype(str).str.extract(r'(\d{5})$').astype(int)
    
    # Sort by rank (ascending = highest ranked first)
    unmatched_recruits_ranked = unmatched_recruits.sort_values('rank')
    
    print(f"\nTotal unmatched recruits: {len(unmatched_recruits)}")
    print(f"\n--- TOP 30 HIGHEST RANKED UNMATCHED RECRUITS ---\n")
    
    # Display top 30
    display_cols = ['year', 'rank', 'name', 'high_school', 'state']
    top_30 = unmatched_recruits_ranked.head(30).copy()
    top_30['rank_str'] = '#' + top_30['rank'].astype(str)
    
    print(top_30[['year', 'rank_str', 'name', 'high_school', 'state']].to_string(index=False, header=['Year', 'Rank', 'Name', 'School', 'State']))
    
    # Summary by year
    print(f"\n--- UNMATCHED RECRUITS BY YEAR ---")
    print(unmatched_recruits['year'].value_counts().sort_index(ascending=False).to_string())
    
    # Distribution of top-ranked unmatched
    print(f"\n--- TOP RECRUIT RANKINGS DISTRIBUTION ---")
    print(f"Unmatched Top 100 recruits (rank 1-100): {len(unmatched_recruits[unmatched_recruits['rank'] <= 100])}")
    print(f"Unmatched Top 250 recruits (rank 1-250): {len(unmatched_recruits[unmatched_recruits['rank'] <= 250])}")
    print(f"Unmatched Top 500 recruits (rank 1-500): {len(unmatched_recruits[unmatched_recruits['rank'] <= 500])}")
    print(f"Unmatched outside Top 500 (rank 501+): {len(unmatched_recruits[unmatched_recruits['rank'] > 500])}")

else:
    print("All recruits have been matched!")


HIGHEST RANKED UNMATCHED RECRUITS

Total unmatched recruits: 37340

--- TOP 30 HIGHEST RANKED UNMATCHED RECRUITS ---

Year Rank               Name               School State
2015   #1   Trenton Thompson             Westover    GA
2019   #1        Nolan Smith          IMG Academy    FL
2016   #1        Rashan Gary     Paramus Catholic    NJ
2020   #1       Bryan Bresee             Damascus    MD
2028   #1      Brysen Wright             Mandarin    FL
2027   #1  John Meredith III        North Crowley    TX
2021   #2      Korey Foreman           Centennial    CA
2026   #2        Lamar Brown       University Lab    LA
2016   #2    Dexter Lawrence          Wake Forest    NC
2028   #2     Jalanie George          Desert Edge    AZ
2027   #2      Mark Matthews   St. Thomas Aquinas    FL
2015   #2        Martez Ivey               Apopka    FL
2024   #2  Ellis Robinson IV          IMG Academy    FL
2027   #3      Joshua Dobson                Hough    NC
2028   #3        A'mir Sears             

In [81]:
# Coverage Analysis by Recruit Position
print("\n" + "="*80)
print("MATCH COVERAGE BY RECRUIT POSITION")
print("="*80)

# Reload recruit data with position information
recruits_with_position = []

for recruit_file in sorted(glob.glob(os.path.join(RECRUITS_DIR, "recruits_*.csv"))):
    try:
        df = pd.read_csv(recruit_file)
        filename = os.path.basename(recruit_file)
        
        # Extract columns including position if available
        available_cols = ['year', 'name', 'high_school', 'state', 'player_id', 'url']
        # Check if position column exists
        if 'position' in df.columns:
            available_cols.append('position')
        
        subset = df[available_cols].copy()
        subset['name_cleaned'] = subset['name'].apply(clean_name)
        recruits_with_position.append(subset)
    except Exception as e:
        pass

if recruits_with_position:
    recruits_pos = pd.concat(recruits_with_position, ignore_index=True)
    
    if 'position' in recruits_pos.columns:
        # Merge matched status
        recruits_pos['is_matched'] = recruits_pos['name'].isin(matched_recruits)
        
        print(f"\n--- MATCH COVERAGE BY POSITION ---\n")
        
        # Create summary by position (counting UNIQUE recruits, not rows which may have duplicates)
        position_summary = []
        for pos in recruits_pos['position'].unique():
            pos_data = recruits_pos[recruits_pos['position'] == pos]
            total_unique = pos_data['name'].nunique()
            matched_unique = pos_data[pos_data['is_matched']]['name'].nunique()
            position_summary.append({
                'position': pos,
                'Total': total_unique,
                'Matched': matched_unique,
                'Coverage %': (matched_unique / total_unique * 100) if total_unique > 0 else 0
            })
        
        position_summary = pd.DataFrame(position_summary)
        position_summary = position_summary.sort_values('Total', ascending=False).reset_index(drop=True)
        position_summary['Coverage %'] = position_summary['Coverage %'].round(1)
        position_summary = position_summary[['Total', 'Matched', 'Coverage %']].sort_values('Total', ascending=False)
        
        print(position_summary.to_string())
        
        # Positions with lowest coverage
        print(f"\n--- POSITIONS WITH LOWEST COVERAGE ---\n")
        low_coverage = position_summary.sort_values('Coverage %').head(10)
        print(low_coverage.to_string())
        
        # Positions with highest unmatched counts
        print(f"\n--- POSITIONS WITH MOST UNMATCHED RECRUITS ---\n")
        position_summary['Unmatched'] = position_summary['Total'] - position_summary['Matched']
        most_unmatched = position_summary.sort_values('Unmatched', ascending=False).head(10)
        print(most_unmatched[['Total', 'Matched', 'Unmatched', 'Coverage %']].to_string())
        
        # Overall coverage (count unique recruits, not rows)
        total_recruits_unique = recruits_pos['name'].nunique()
        matched_count = recruits_pos[recruits_pos['is_matched']]['name'].nunique()
        overall_rate = 100 * matched_count / total_recruits_unique
        
        print(f"\n--- OVERALL SUMMARY ---")
        print(f"Total Unique Recruits: {total_recruits_unique:,}")
        print(f"Matched (Unique): {matched_count:,}")
        print(f"Unmatched (Unique): {total_recruits_unique - matched_count:,}")
        print(f"Overall Match Rate: {overall_rate:.1f}%")
    else:
        print("Position column not found in recruit data")
else:
    print("Could not load recruit position data")


MATCH COVERAGE BY RECRUIT POSITION

--- MATCH COVERAGE BY POSITION ---

    Total  Matched  Coverage %
0    5324      984        18.5
1    3591      299         8.3
2    3369      428        12.7
3    3312       40         1.2
4    3164      265         8.4
5    2801      432        15.4
6    1937       87         4.5
7    1851      125         6.8
8    1726      201        11.6
9    1589      125         7.9
10   1563        4         0.3
11   1422       57         4.0
12   1360        9         0.7
13   1245      158        12.7
14   1243      452        36.4
15   1119      130        11.6
16   1074       68         6.3
17   1061      104         9.8
18    948      340        35.9
19    588      153        26.0
20    374        6         1.6
21    354        7         2.0
22    303       47        15.5
23    216        3         1.4
24    137        3         2.2
25     62        8        12.9
26      1        0         0.0

--- POSITIONS WITH LOWEST COVERAGE ---

    Total  Matched

In [82]:
# Top 10 Unmatched Recruits by Position (QB, PRO, WR, RB)
print("\n" + "="*80)
print("TOP 10 UNMATCHED RECRUITS BY POSITION")
print("="*80)

# Use recruits_pos if available (has position data), otherwise reload with positions
if 'recruits_pos' not in locals() or recruits_pos is None:
    print("Loading recruit data with positions...")
    recruits_pos_data = []
    for recruit_file in sorted(glob.glob(os.path.join(RECRUITS_DIR, "recruits_*.csv"))):
        try:
            df = pd.read_csv(recruit_file)
            available_cols = ['year', 'name', 'high_school', 'state', 'player_id', 'url']
            if 'position' in df.columns:
                available_cols.append('position')
            subset = df[available_cols].copy()
            subset['name_cleaned'] = subset['name'].apply(clean_name)
            recruits_pos_data.append(subset)
        except Exception as e:
            pass
    
    if recruits_pos_data:
        recruits_pos = pd.concat(recruits_pos_data, ignore_index=True)
    else:
        recruits_pos = None

# Filter for key positions
key_positions = ['QB', 'PRO', 'WR', 'RB']

if recruits_pos is not None and 'position' in recruits_pos.columns:
    # Get unmatched recruits with position info
    recruits_pos['is_matched'] = recruits_pos['name'].isin(matched_recruits)
    
    for position in key_positions:
        print(f"\n{'='*80}")
        print(f"{position} - Top 10 Unmatched Ranked Recruits")
        print(f"{'='*80}\n")
        
        # Filter unmatched recruits for this position
        pos_unmatched = recruits_pos[(recruits_pos['position'] == position) & (~recruits_pos['is_matched'])].copy()
        
        if len(pos_unmatched) > 0:
            # Extract rank from player_id and sort
            pos_unmatched['rank'] = pos_unmatched['player_id'].astype(str).str.extract(r'(\d{5})$').astype(int)
            pos_unmatched_ranked = pos_unmatched.sort_values('rank').head(10)
            
            # Format and display
            display_df = pos_unmatched_ranked[['year', 'rank', 'name', 'high_school', 'state']].copy()
            display_df['rank'] = '#' + display_df['rank'].astype(str)
            display_df.columns = ['Year', 'Rank', 'Name', 'School', 'State']
            
            print(display_df.to_string(index=False))
            
            # Summary for position
            total_pos = len(recruits_pos[recruits_pos['position'] == position])
            matched_pos = len(recruits_pos[(recruits_pos['position'] == position) & (recruits_pos['is_matched'])])
            unmatched_pos = total_pos - matched_pos
            
            if total_pos > 0:
                rate = 100 * matched_pos / total_pos
                print(f"\nSummary: {unmatched_pos} unmatched out of {total_pos} {position} recruits ({rate:.1f}% coverage)")
        else:
            total_pos = len(recruits_pos[recruits_pos['position'] == position])
            matched_pos = len(recruits_pos[(recruits_pos['position'] == position) & (recruits_pos['is_matched'])])
            rate = 100 * matched_pos / total_pos if total_pos > 0 else 0
            print(f"All {position} recruits matched! ({total_pos} total, {rate:.1f}% coverage)")
else:
    print("Position data not available in recruit database")



TOP 10 UNMATCHED RECRUITS BY POSITION

QB - Top 10 Unmatched Ranked Recruits

 Year Rank               Name             School State
 2023   #3     Nico Iamaleava             Warren    CA
 2023   #4        Dante Moore Martin Luther King    MI
 2028   #4        Jayden Wade        IMG Academy    FL
 2025   #7   Tavien St. Clair      Bellefontaine    OH
 2028   #7 Christopher Vargas    St. John's Prep    MA
 2024   #7          DJ Lagway             Willis    TX
 2026   #9           Dia Bell  American Heritage    FL
 2023  #11     Malachi Nelson       Los Alamitos    CA
 2021  #25      J.J. McCarthy        IMG Academy    IL
 2022  #26         Ty Simpson           Westview    TN

Summary: 793 unmatched out of 1245 QB recruits (36.3% coverage)

PRO - Top 10 Unmatched Ranked Recruits

 Year Rank             Name            School State
 2016   #4   Shea Patterson       IMG Academy    FL
 2016  #49      Malik Henry   Long Beach Poly    CA
 2018  #63      Matt Corral   Long Beach Poly    CA
 2

In [77]:
# DEBUG: Check for discrepancies between saved file and coverage analysis
print("\n" + "="*80)
print("DIAGNOSTIC: MATCH COUNT VERIFICATION")
print("="*80)

# Check actual file
import os
csv_path = os.path.join(NAME_MATCH_DIR, "hs_to_recruit_all_matches.csv")
if os.path.exists(csv_path):
    all_matches_file = pd.read_csv(csv_path)
    print(f"\nActual CSV file rows: {len(all_matches_file)}")
    print(f"Unique recruit names in CSV: {all_matches_file['recruit_name'].nunique()}")
else:
    print(f"CSV file not found at {csv_path}")

# Check in-memory variables
print(f"\nIn-memory all_matches rows: {len(all_matches)}")
print(f"Unique recruit names in memory: {len(matched_recruits)}")

# Check recruits_pos
print(f"\nTotal rows in recruits_pos: {len(recruits_pos)}")
print(f"Unique recruit names in recruits_pos: {recruits_pos['name'].nunique()}")
print(f"Rows marked as matched in recruits_pos: {recruits_pos['is_matched'].sum()}")

# Find duplicates in recruits_pos
print(f"\nDuplicate recruit names in recruits_pos:")
dup_names = recruits_pos['name'].value_counts()
dup_names = dup_names[dup_names > 1]
if len(dup_names) > 0:
    print(f"  {len(dup_names)} recruits appear multiple times")
    print(f"  Example: {dup_names.head()}")
else:
    print("  No duplicates found")

# The issue is likely that recruits can appear multiple times (different years)
# Let's check the actua unique coverage
unique_matched_in_pos = recruits_pos[recruits_pos['is_matched']]['name'].nunique()
print(f"\nUnique matched recruit names actually in recruits_pos: {unique_matched_in_pos}")



DIAGNOSTIC: MATCH COUNT VERIFICATION

Actual CSV file rows: 4241
Unique recruit names in CSV: 4158

In-memory all_matches rows: 4241
Unique recruit names in memory: 4158

Total rows in recruits_pos: 41926
Unique recruit names in recruits_pos: 40340
Rows marked as matched in recruits_pos: 4586

Duplicate recruit names in recruits_pos:
  1152 recruits appear multiple times
  Example: name
Jalen Williams     9
Brandon Jones      8
Jordan Williams    8
Jordan Davis       7
Tyler Johnson      7
Name: count, dtype: int64

Unique matched recruit names actually in recruits_pos: 4158


## Output Files

The matcher has generated the following files in `data/name_matching/`:

1. **hs_to_recruit_exact_matches.csv** - Layer 1: Exact name + exact state + fuzzy school (≥80%)
2. **hs_to_recruit_fuzzy_matches.csv** - Layer 2: Name variations + fuzzy name (≥90%) + exact state + fuzzy school (≥80%)
3. **hs_to_recruit_all_matches.csv** - Combined results from all layers

Each output file contains:
- `hs_player`, `hs_school`, `hs_state`, `hs_year` - High school player info
- `recruit_name`, `recruit_school`, `recruit_state`, `recruit_year` - Matched recruit info
- `recruit_id` - Player ID for linking to recruit database
- `match_confidence` - Confidence score (0-100)
- `matching_method` - Which layer found the match

**School Name Cleaning:**
All school names have been cleaned during preprocessing by removing "HS" and "High School" indicators, enabling more reliable fuzzy matching.

**Next Steps:**
1. Review the unmatched players to identify remaining patterns
2. Use the `hs_to_recruit_all_matches.csv` to create linkage keys
3. Connect back to the existing recruit → college mapping using `recruit_id`